# Published Figures — configurable driver

A single, editable front-end over `acute_slice_mea.published_figure`, the same
functions used by `scripts/make_panel_a_figure.py`, `make_panel_b_figure.py`,
and `make_combined_figure.py`.

This notebook builds **three** figures:

* **Panel A** — activity-scan map + RMS-coloured network electrodes (ROI bold).
* **Panel B** — ROI LFP waveform snapshot + per-electrode robust-SD raster.
* **Combined** — Panel A stacked over Panel B.

**The only cell you normally edit is the _Configuration_ cell below.** Change
the image (SVG) path, raw `.h5` path, ROI electrodes, time windows, crop, output
location, etc., then run all cells. Each figure renders inline **and** is saved
as `<out_stem>.svg` + `<out_stem>.png`.

Defaults reproduce the committed CarenPaula plate-16719 figures.

## Setup (imports + editable `src` on path)

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path
import sys

project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
src_path = project_root / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
# If acute_slice_mea was imported from elsewhere (e.g. an editable install in a
# different checkout), evict it so this repo's src/ wins.
loaded_package = sys.modules.get("acute_slice_mea")
loaded_package_file = getattr(loaded_package, "__file__", None)
if loaded_package_file and src_path.resolve() not in Path(loaded_package_file).resolve().parents:
    for module_name in [name for name in sys.modules if name == "acute_slice_mea" or name.startswith("acute_slice_mea.")]:
        del sys.modules[module_name]

from acute_slice_mea.published_figure import (
    compute_roi_rms,
    make_panel_a_figure,
    make_panel_b_figure,
    make_combined_figure,
)

print(f"project_root = {project_root}")

## Configuration (edit me)

Every knob the three scripts expose, in one place. Paths are absolute (the
notebook's working directory is `notebooks/`). Outputs and the RMS cache are
anchored to `project_root` so they land in the repo's `figures/` and
`data/processed/` regardless of where Jupyter is launched.

In [ ]:
# --- Dataset paths -------------------------------------------------------
# Base plate directory; SVG_PATH / H5_PATH are derived from it but may be
# overridden directly with any absolute path.
PLATE = (
    "/mnt/benshalom-nas/raw_data/irc_maxone_desktop/home/mxwbio/Data2/"
    "MeaSlices_CarenPaula_04082026/MeaSlices_CarenPaula_04082026/260408/16719"
)
SVG_PATH = f"{PLATE}/ActivityScan/000021/analysis/ActivityAnalysis_v1/0001/Well1/amplitudeMap.svg"  # activity-map image (Panel A background)
H5_PATH = f"{PLATE}/Network/000018/data.raw.h5"  # network-scan raw data (RMS + LFP source)
WELL_ID = None  # MaxWell stream/well id; None = auto-detect single well

# --- Region of interest --------------------------------------------------
ROI_IDS = [63, 91, 115, 272]  # electrode_ids: bold border + one waveform/raster row each

# --- Time windows --------------------------------------------------------
RMS_START_SEC = 66.0    # RMS window start (Panel A colouring)
RMS_WINDOW_SEC = 10.0   # RMS window length
SNAP_START_SEC = 66.0   # waveform/raster snapshot start (Panel B)
SNAP_WINDOW_SEC = 1.0   # snapshot length
THRESHOLD_SD = 3.0      # raster threshold in robust SD (MAD-derived)

# --- Spatial / display ---------------------------------------------------
CROP = None             # (x0, x1, y0, y1) microns; None = full chip
INVERT_Y = True         # True puts y=0 at top (MaxWell chip convention)
IMAGE_ORIGIN = "lower"  # imshow origin for the background bitmap: 'lower' or 'upper'
DPI = 300               # PNG export resolution

# --- Outputs -------------------------------------------------------------
OUT_DIR = project_root / "figures"          # where .svg/.png are written
PANEL_A_STEM = "published_panel_a"          # -> <OUT_DIR>/<stem>.{svg,png}
PANEL_B_STEM = "published_panel_b"
COMBINED_STEM = "published_combined"

# --- RMS cache -----------------------------------------------------------
# The per-electrode RMS table is cached so re-running to tweak styling is cheap.
CACHE_PATH = project_root / "data/processed/16719_net000018_cache/electrodes_rms.csv"
FORCE_RMS = False  # True = recompute RMS, ignore cache (set after changing dataset / RMS window)

# --- Which figures to build ---------------------------------------------
RUN_PANEL_A = True
RUN_PANEL_B = True
RUN_COMBINED = True

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"SVG_PATH = {SVG_PATH}")
print(f"H5_PATH  = {H5_PATH}")
print(f"OUT_DIR  = {OUT_DIR}")

## Compute / cache RMS

Computed once and reused by Panel A and the Combined figure (the heavy `.h5`
read happens here; the scripts inject this same `rms_table` to avoid repeating
it). Cached to `CACHE_PATH`; set `FORCE_RMS = True` above to recompute.

In [ ]:
rms_table = compute_roi_rms(
    H5_PATH,
    well_id=WELL_ID,
    start_sec=RMS_START_SEC,
    window_sec=RMS_WINDOW_SEC,
    cache_path=CACHE_PATH,
    force=FORCE_RMS,
)
rms_table.head()

## Panel A — activity map + RMS electrodes

In [ ]:
if RUN_PANEL_A:
    fig_a = make_panel_a_figure(
        SVG_PATH,
        H5_PATH,
        ROI_IDS,
        OUT_DIR / PANEL_A_STEM,
        crop=CROP,
        start_sec=RMS_START_SEC,
        window_sec=RMS_WINDOW_SEC,
        well_id=WELL_ID,
        rms_table=rms_table,
        invert_y=INVERT_Y,
        image_origin=IMAGE_ORIGIN,
        dpi=DPI,
    )
    print(f"Wrote {OUT_DIR / PANEL_A_STEM}.svg and .png")

## Panel B — ROI waveform snapshot + robust-SD raster

In [ ]:
if RUN_PANEL_B:
    fig_b = make_panel_b_figure(
        H5_PATH,
        ROI_IDS,
        OUT_DIR / PANEL_B_STEM,
        start_sec=SNAP_START_SEC,
        window_sec=SNAP_WINDOW_SEC,
        well_id=WELL_ID,
        threshold_sd=THRESHOLD_SD,
        dpi=DPI,
    )
    print(f"Wrote {OUT_DIR / PANEL_B_STEM}.svg and .png")

## Combined — Panel A stacked over Panel B

In [ ]:
if RUN_COMBINED:
    fig_combined = make_combined_figure(
        SVG_PATH,
        H5_PATH,
        ROI_IDS,
        OUT_DIR / COMBINED_STEM,
        crop=CROP,
        rms_start_sec=RMS_START_SEC,
        rms_window_sec=RMS_WINDOW_SEC,
        snap_start_sec=SNAP_START_SEC,
        snap_window_sec=SNAP_WINDOW_SEC,
        threshold_sd=THRESHOLD_SD,
        well_id=WELL_ID,
        rms_table=rms_table,
        invert_y=INVERT_Y,
        image_origin=IMAGE_ORIGIN,
        dpi=DPI,
    )
    print(f"Wrote {OUT_DIR / COMBINED_STEM}.svg and .png")

## Where the files went

Each enabled section wrote `<out_stem>.svg` and `<out_stem>.png` under
`OUT_DIR` (default `figures/`):

* `published_panel_a.{svg,png}`
* `published_panel_b.{svg,png}`
* `published_combined.{svg,png}`

**Cheap restyling:** the RMS table is cached at `CACHE_PATH`, so re-running the
figure cells after a styling tweak does *not* re-read the `.h5`. Set
`FORCE_RMS = True` (and re-run the RMS cell) only when you change the dataset
(`H5_PATH`/`WELL_ID`) or the RMS window (`RMS_START_SEC`/`RMS_WINDOW_SEC`).